In [17]:
import pandas as pd
import numpy as np
import torch


In [2]:
train_df = pd.read_parquet("train.parquet")

In [3]:
train_df.head(5)

,pubid,question,context,long_answer,final_decision,combined_context
616,9347843,Laparoscopic-assisted ileocolic resections in ...,{'contexts': ['Because of the inflammatory nat...,The laparoscopic-assisted approach to Crohn's ...,no,Because of the inflammatory nature of Crohn's ...
684,21789019,Do elderly cancer patients have different care...,{'contexts': ['The increasingly older populati...,Elderly patients have informational and relati...,no,The increasingly older population confronts on...
208,26452334,Measurement of head and neck paragangliomas: i...,{'contexts': ['The aim of this study was to as...,"Due to a relatively good reproducibility, fast...",yes,The aim of this study was to assess the reprod...
952,23517744,Is solitary kidney really more resistant to is...,{'contexts': ['To our knowledge there are no e...,Solitary kidney in a canine model is more resi...,yes,To our knowledge there are no evidence-based m...
988,11438275,Does patient position during liver surgery inf...,{'contexts': ['It is generally believed that p...,The effect on venous pressures caused by the c...,no,It is generally believed that positioning of t...


In [4]:
train_df["Question and Context"]  = train_df["question"] + train_df["combined_context"]

In [5]:
train_df["Question and Context"].shape

(800,)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
X_tfidf =  vectorizer.fit_transform(train_df["Question and Context"])

In [7]:
!pip install gensim

In [8]:
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')

tokenized_data = train_df["Question and Context"].apply(word_tokenize)

[nltk_data] Downloading package punkt to C:\Users\Aman Kumar
[nltk_data]     Singh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [9]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=tokenized_data,
    vector_size=100,   # embedding dimension
    window=5,          # context size
    min_count=2,       # ignore rare words
    workers=4,         # parallel processing
    sg=1               # 1 = Skip-gram, 0 = CBOW
)

In [10]:
!pip install --upgrade gensim numpy

  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata (6.6 kB)


In [12]:
def document_vector(doc):
    temp_list = [model.wv[i] for i in doc if i in model.wv]
    doc_vec = np.mean(temp_list, axis = 0)
    return doc_vec

In [13]:
tokenized_data

616    [Laparoscopic-assisted, ileocolic, resections,...
684    [Do, elderly, cancer, patients, have, differen...
208    [Measurement, of, head, and, neck, paraganglio...
952    [Is, solitary, kidney, really, more, resistant...
988    [Does, patient, position, during, liver, surge...
                             ...                        
918    [Should, displaced, midshaft, clavicular, frac...
418    [Evaluation, of, pediatric, VCUG, at, an, acad...
65     [Should, circumcision, be, performed, in, chil...
159    [Regional, anesthesia, as, compared, with, gen...
509    [Are, polymorphisms, in, oestrogen, receptors,...
Name: Question and Context, Length: 800, dtype: object

In [15]:
X_wv = tokenized_data.apply(document_vector)

In [20]:
from transformers import AutoTokenizer, AutoModel
MODEL_NAME = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

model.eval()
def document(text):
    encoded = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )
    
    with torch.no_grad():
        outputs = model(**encoded)

    cls_embedding = outputs.last_hidden_state[:, 0, :]

    return cls_embedding.squeeze().numpy()


pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

In [21]:
X_bert = train_df["Question and Context"].apply(document)

In [26]:
len(X_bert[1])

768